importing necessary libraries

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "iframe"
import plotly.express as px
from textblob import TextBlob

In [ ]:
df = pd.read_csv("/Users/abhimanyuchettiar/Downloads/spotify-2023.csv", encoding='latin-1')

overview of the dataset

In [ ]:
df.head()

data cleaning

In [ ]:
# 1. Convert 'streams' to numeric. 
# 'errors=coerce' turns any non-numeric text (like 'BPM...') into NaN (Not a Number)
df['streams'] = pd.to_numeric(df['streams'], errors='coerce')

# 2. Clean columns that have commas in them
# We strip the comma and convert to a number
for col in ['in_deezer_playlists', 'in_shazam_charts']:
    df[col] = df[col].astype(str).str.replace(',', '')
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Remove rows where 'streams' is empty
df = df.dropna(subset=['streams'])

# 4. Fill missing musical 'keys' with a placeholder
df['key'] = df['key'].fillna('Unknown')

Top 10 Most Streamed Songs

In [ ]:
top_10 = df.nlargest(10, 'streams')

fig1 = px.bar(top_10, x='streams', y='track_name', orientation='h',
             title='1. Top 10 Most Streamed Songs (2023)',
             color='streams', color_continuous_scale='Viridis')
fig1.show()

Artist Dominance

In [ ]:
# Count hits per artist
top_artists = df['artist(s)_name'].str.split(', ').explode().value_counts().head(10).reset_index()
top_artists.columns = ['Artist', 'Hit Count']

fig1 = px.bar(top_artists, x='Artist', y='Hit Count', 
             title='Who Owns the Charts? (Top 10 Artist Dominance)',
             color='Hit Count', text_auto=True)
fig1.show()

Mood map - Energy vs Valence, this tells us if the most popular songs are "happy and energetic" or "sad and calm"

In [ ]:
fig3 = px.scatter(df, 
                 x='valence_%', 
                 y='energy_%', 
                 color='danceability_%',
                 hover_data=['track_name', 'artist(s)_name'],
                 title='Mood Map: Energy vs Valence (Positivity)',
                 labels={'valence_%': 'Valence (Happiness %)', 'energy_%': 'Energy (%)'},
                 color_continuous_scale='RdYlGn')
fig3.show()


Feature Correlation Heatmap, tells us which musical traits go together

In [ ]:
features = ['danceability_%', 'valence_%', 'energy_%', 'acousticness_%', 'speechiness_%']
corr = df[features].corr()

fig4 = px.imshow(corr, text_auto=True, title='4. Correlation of Musical Features',
                color_continuous_scale='RdBu_r')
fig4.show()

Seasonality: When are Hits released?

In [ ]:
# Map numbers to names for better reading
month_map = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 
             7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
monthly = df.groupby('released_month').size().reset_index(name='Count')
monthly['Month'] = monthly['released_month'].map(month_map)

fig5 = px.line(monthly, x='Month', y='Count', markers=True, 
              title='5. Number of Hits Released by Month')
fig5.show()

Platform Wars: Spotify vs Apple Music, compare how a song's performance on the Spotify Charts correlates with its Apple Music Chart Rank

In [ ]:
fig3 = px.scatter(df, x='in_spotify_charts', y='in_apple_charts', 
                 trendline="ols", 
                 hover_data=['track_name'],
                 title='Spotify vs. Apple: Do they agree on what is a hit?')
fig3.show()

Compare top 50 most streamed songs against the average of the whole dataset

In [ ]:
# Calculate averages
avg_all = df[['danceability_%', 'valence_%', 'energy_%', 'acousticness_%']].mean()
avg_top50 = df.nlargest(50, 'streams')[['danceability_%', 'valence_%', 'energy_%', 'acousticness_%']].mean()

# Combine for plotting
comparison = pd.DataFrame({'Metric': avg_all.index, 'Overall Avg': avg_all.values, 'Top 50 Avg': avg_top50.values})

fig4 = px.bar(comparison, x='Metric', y=['Overall Avg', 'Top 50 Avg'], barmode='group',
             title='Anatomy of a Megahit: Top 50 vs. The Rest')
fig4.show()